# 20b · Quantization and rescoring

## The engineering question
How much fidelity and storage does scalar quantization trade for search efficiency, and what can rescoring repair?

**Learning outcomes:** distinguish encoder quantization from stored-vector quantization; compare exact full precision, ANN, INT8 candidates, and full-vector rescoring; record index readiness, physical bytes, latency, and semantic quality separately.

Qdrant 1.17.1 stores float32 originals plus an optional INT8 representation. Compression of the search representation does not mean total collection disk usage shrinks fourfold. Rescoring reads originals for an oversampled candidate set; it cannot rescue an absent candidate. The experiment contains 6,000 synthetic vectors for search fidelity and separately tagged real course-document embeddings for semantic evaluation. Synthetic recall is never a relevance score.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("20b", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
from uuid import uuid4
from qdrant_client import QdrantClient, models as qm
from vector_masterclass.config import qdrant_url
from vector_masterclass.operations import wait_index, collection_storage_bytes
from vector_masterclass.models import embed
from vector_masterclass.data import documents, queries
from vector_masterclass.metrics import ranking_metrics, ann_recall

rng = np.random.default_rng(731)
synthetic = rng.normal(size=(6000, 384)).astype("float32")
synthetic /= np.linalg.norm(synthetic, axis=1, keepdims=True)
query_vectors = rng.normal(size=(20, 384)).astype("float32")
query_vectors /= np.linalg.norm(query_vectors, axis=1, keepdims=True)
docs = documents(tenant="public")
semantic = embed([d.text for d in docs])
vectors = np.vstack([synthetic, semantic])
client = QdrantClient(url=qdrant_url(), timeout=30)
name = "vmc_20b_" + uuid4().hex[:8]
client.create_collection(
    name,
    vectors_config=qm.VectorParams(size=384, distance=qm.Distance.COSINE, on_disk=True),
    hnsw_config=qm.HnswConfigDiff(m=16, ef_construct=100, full_scan_threshold=10),
    optimizers_config=qm.OptimizersConfigDiff(indexing_threshold=64, default_segment_number=1),
    quantization_config=qm.ScalarQuantization(
        scalar=qm.ScalarQuantizationConfig(type=qm.ScalarType.INT8, quantile=0.99, always_ram=True)
    ),
)
rows, quality = [], []
configs = {
    "float32 exact": qm.SearchParams(
        exact=True, quantization=qm.QuantizationSearchParams(ignore=True)
    ),
    "float32 ANN": qm.SearchParams(
        hnsw_ef=64, quantization=qm.QuantizationSearchParams(ignore=True)
    ),
    "INT8 no rescore": qm.SearchParams(
        hnsw_ef=64, quantization=qm.QuantizationSearchParams(rescore=False)
    ),
    "INT8 rescore x3": qm.SearchParams(
        hnsw_ef=64, quantization=qm.QuantizationSearchParams(rescore=True, oversampling=3.0)
    ),
}
try:
    started = time.perf_counter()
    for start in range(0, len(vectors), 256):
        client.upsert(
            name,
            points=[
                qm.PointStruct(
                    id=i,
                    vector=vectors[i].tolist(),
                    payload={
                        "kind": "synthetic" if i < 6000 else "semantic",
                        "source": None if i < 6000 else docs[i - 6000].key,
                    },
                )
                for i in range(start, min(start + 256, len(vectors)))
            ],
            wait=True,
        )
    info = wait_index(client, name, len(vectors))
    build_seconds = time.perf_counter() - started
    physical_bytes = collection_storage_bytes(name)
    for method, params in configs.items():
        client.query_points(name, query=query_vectors[0].tolist(), limit=10, search_params=params)
        for qi, vector in enumerate(query_vectors):
            oracle = np.argsort(-(vectors @ vector), kind="stable")[:10].tolist()
            started = time.perf_counter()
            hits = client.query_points(
                name, query=vector.tolist(), limit=10, search_params=params
            ).points
            rows.append(
                {
                    "method": method,
                    "query": qi,
                    "ann_recall": ann_recall([h.id for h in hits], oracle, 10),
                    "milliseconds": 1000 * (time.perf_counter() - started),
                }
            )
        for q, vector in zip(
            queries("development"), embed([q.text for q in queries("development")], role="query")
        ):
            hits = client.query_points(
                name,
                query=vector.tolist(),
                query_filter=qm.Filter(
                    must=[qm.FieldCondition(key="kind", match=qm.MatchValue(value="semantic"))]
                ),
                limit=5,
                search_params=params,
            ).points
            quality.append(
                {
                    "method": method,
                    "query": q.id,
                    **ranking_metrics([h.payload["source"] for h in hits], q.relevance, 5),
                }
            )
finally:
    client.delete_collection(name)
    client.close()
results = lesson.table(pd.DataFrame(rows))
quality = lesson.table(pd.DataFrame(quality), "semantic-quality")
summary = results.groupby("method").agg(
    fidelity=("ann_recall", "mean"),
    median_ms=("milliseconds", "median"),
    p95_ms=("milliseconds", lambda x: x.quantile(0.95)),
)
display(summary)
display(quality.groupby("method")[["recall", "mrr", "ndcg"]].mean())
display(
    {
        "raw_float32_bytes": vectors.nbytes,
        "nominal_INT8_bytes": vectors.size,
        "measured_collection_allocated_bytes": physical_bytes,
        "build_seconds": build_seconds,
        "indexed_vectors": info.indexed_vectors_count,
    }
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
summary.fidelity.plot.bar(ax=axes[0], ylim=(0, 1.05), title="Synthetic search fidelity@10")
summary.median_ms.plot.bar(ax=axes[1], title="Warm median request latency", ylabel="Milliseconds")
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Inspect fidelity and semantic results independently. This tiny semantic filter may lead to a scan; it is not evidence of a large-corpus ANN advantage. The synthetic Gaussian distribution is convenient for controlling dimensions but differs from trained embedding distributions. Quantization wins, ties, or regressions are retained.

**Exercise:** vary oversampling on development inputs while keeping `hnsw_ef` fixed. Explain the extra full-vector reads and why a larger candidate set can restore recall. Compare physical collection bytes with the float32 and INT8 arithmetic; account for originals, HNSW edges, payloads, WAL, and segment files.

Encoder weight quantization is a different experiment: it can change the embeddings themselves and requires rebuilding or validating compatibility. Current Qdrant documentation can describe newer APIs; this notebook deliberately uses the tested 1.17.1 `on_disk`/`always_ram` contract. [Qdrant quantization concepts](https://qdrant.tech/documentation/manage-data/quantization/).

In [ ]:
manifest = lesson.finish(
    {
        "four_search_modes": results.method.nunique() == 4,
        "eighty_requests": len(results) == 80,
        "exact_matches_numpy": bool(
            (results.loc[results.method == "float32 exact", "ann_recall"] == 1).all()
        ),
        "physical_bytes_measured": physical_bytes > 0,
        "semantic_quality_separate": len(quality) == 28,
    },
    observations={
        "build_seconds": build_seconds,
        "collection_allocated_bytes": physical_bytes,
        "raw_float32_bytes": vectors.nbytes,
        "nominal_INT8_bytes": vectors.size,
        "index_state": info.model_dump(mode="json"),
        "synthetic_queries": 20,
        "semantic_split": "development",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.